# 模块二：在 Google Colab T4 上训练 SD 1.5 LoRA

这个 Notebook 会用已经准备好的 `module_02_colab_dataset_v4.zip` 微调现成的 Stable Diffusion 1.5。我们训练的是轻量 LoRA，不是从头训练整个模型。

运行前：在 Colab 菜单选择 **代码执行程序 → 更改运行时类型 → T4 GPU**，然后从上到下依次运行。首次完整训练通常约需 1～2 小时，下载模型的时间另算。

## 1. 检查 GPU

这一格必须看到类似 `Tesla T4`，否则不要开始训练。

In [ ]:
import torch

assert torch.cuda.is_available(), "没有检测到 GPU。请在菜单中选择 T4 GPU 后重新运行。"
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"显存: {gpu_memory_gb:.1f} GB")
assert gpu_memory_gb >= 14, "当前显存低于 14 GB，这套参数可能无法稳定运行。"
!nvidia-smi

## 2. 挂载 Google Drive

训练检查点和最终权重会保存在 `我的云端硬盘/lighting_lora/`。Colab 要求授权时，按页面提示登录即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 上传数据 ZIP

点击运行后选择本地的 `module_02_colab_dataset_v4.zip`。上传完成后，Notebook 会同时备份一份到 Google Drive。

In [ ]:
from pathlib import Path
from google.colab import files
import hashlib
import shutil

PROJECT_DIR = Path('/content/drive/MyDrive/lighting_lora')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_NAME = 'module_02_colab_dataset_v4.zip'
DRIVE_ZIP = PROJECT_DIR / ZIP_NAME
EXPECTED_ZIP_SHA256 = 'b8d6e3ed2ac2c46fa3923f255fb0577190b2e12e5f8eca944d3af0984b6c7840'

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

drive_zip_is_current = DRIVE_ZIP.exists() and file_sha256(DRIVE_ZIP) == EXPECTED_ZIP_SHA256
if not drive_zip_is_current:
    if DRIVE_ZIP.exists():
        print('云端数据包是旧版本，请重新选择本地 ZIP；上传后将自动覆盖。')
    uploaded = files.upload()
    assert ZIP_NAME in uploaded, f'请选择名为 {ZIP_NAME} 的文件。'
    uploaded_path = Path('/content') / ZIP_NAME
    assert file_sha256(uploaded_path) == EXPECTED_ZIP_SHA256, '所选 ZIP 不是当前正确版本，请重新下载本项目中的 ZIP。'
    shutil.copy2(f'/content/{ZIP_NAME}', DRIVE_ZIP)
    print(f'已备份到：{DRIVE_ZIP}')
else:
    print(f'云端数据包校验通过：{DRIVE_ZIP}')

## 4. 安装官方训练代码

这里固定使用 Hugging Face Diffusers `v0.39.0`：训练脚本来自同版本官方仓库，Python 包使用 PyPI 的完整 wheel。单元格会清除 Colab 中可能残留的同名目录和混装版本，并检查 `diffusers.guiders` 是否完整。

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version
import shutil
import site
import subprocess
import sys
from pathlib import Path

DIFFUSERS_VERSION = '0.39.0'
DIFFUSERS_REPO = Path('/content/hf_diffusers_repo')

# 清理旧 Notebook 可能留下的源码目录，避免 /content/diffusers 遮蔽正式安装包。
for stale_dir in (Path('/content/diffusers'), DIFFUSERS_REPO):
    shutil.rmtree(stale_dir, ignore_errors=True)

subprocess.run(
    [
        'git', '-c', 'advice.detachedHead=false', 'clone', '--quiet', '--depth', '1',
        '--branch', f'v{DIFFUSERS_VERSION}',
        'https://github.com/huggingface/diffusers.git', str(DIFFUSERS_REPO),
    ],
    check=True,
)

# 删除 editable 安装及其残留，再安装官方完整 wheel。
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'diffusers'],
    check=False,
)
for package_root in map(Path, site.getsitepackages()):
    shutil.rmtree(package_root / 'diffusers', ignore_errors=True)
    for metadata_dir in package_root.glob('diffusers-*.dist-info'):
        shutil.rmtree(metadata_dir, ignore_errors=True)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--quiet', '--no-cache-dir',
        '--force-reinstall', '--no-deps', f'diffusers=={DIFFUSERS_VERSION}',
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--quiet', '-r',
        str(DIFFUSERS_REPO / 'examples/text_to_image/requirements.txt'),
    ],
    check=True,
)

# Colab 可能预装旧版 torchao，它会与新版 PEFT 冲突；本项目不需要 torchao。
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'],
    check=False,
)

# 如果当前内核曾加载过旧版，清除模块缓存后再验证新安装。
for module_name in list(sys.modules):
    if module_name == 'diffusers' or module_name.startswith('diffusers.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

try:
    installed_torchao = version('torchao')
    raise RuntimeError(f'torchao {installed_torchao} 仍然存在，请重启运行时后重新执行。')
except PackageNotFoundError:
    pass
for package in ('diffusers', 'accelerate', 'datasets', 'peft'):
    print(f'{package}: {version(package)}')

import diffusers
diffusers_dir = Path(diffusers.__file__).parent
guiders_init = diffusers_dir / 'guiders' / '__init__.py'
assert diffusers.__version__ == DIFFUSERS_VERSION, f'Diffusers 版本错误：{diffusers.__version__}'
assert guiders_init.is_file(), f'Diffusers 安装不完整，缺少：{guiders_init}'
from diffusers import StableDiffusionPipeline
print(f'依赖安装与完整性检查通过：{diffusers_dir}')

## 5. 解压并检查数据

正确结果应为：训练集 254 张、验证集 42 张；相同配方或原始场景组不会跨分区。

In [ ]:
import json
import shutil
import zipfile
from datasets import load_dataset

WORK_DIR = Path('/content/lighting_training')
DATA_ROOT = WORK_DIR / 'colab_dataset'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_ZIP) as archive:
    archive.extractall(WORK_DIR)

assert (DATA_ROOT / 'train' / 'metadata.jsonl').exists(), 'ZIP 内缺少训练集 metadata.jsonl。'
assert (DATA_ROOT / 'validation' / 'metadata.jsonl').exists(), 'ZIP 内缺少验证集 metadata.jsonl。'
for split_name in ('train', 'validation'):
    split_dir = DATA_ROOT / split_name
    rows = [json.loads(line) for line in (split_dir / 'metadata.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    missing = [row['file_name'] for row in rows if not (split_dir / row['file_name']).is_file()]
    assert not missing, f'{split_name} 元数据引用了不存在的图片：{missing[:5]}'
    image_path_fields = sorted({key for row in rows for key in row if key == 'file_name' or key.endswith('_file_name')})
    assert image_path_fields == ['file_name'], f'{split_name} 存在会被 ImageFolder 误识别的图片字段：{image_path_fields}'

dataset = load_dataset('imagefolder', data_dir=str(DATA_ROOT), cache_dir=str(WORK_DIR / 'check_cache'))
print(dataset)
assert len(dataset['train']) == 254, f"训练集数量不对：{len(dataset['train'])}"
assert len(dataset['validation']) == 42, f"验证集数量不对：{len(dataset['validation'])}"
expected_columns = ['image', 'text', 'source', 'scene_prompt', 'original_name', 'content_sha256', 'caption_source', 'caption_model', 'caption_provider']
missing_columns = sorted(set(expected_columns) - set(dataset['train'].column_names))
assert not missing_columns, f"数据缺少必需列：{missing_columns}；实际列：{dataset['train'].column_names}"
for split_name in ('train', 'validation'):
    for item in dataset[split_name]:
        item['image'].load()
print('数据检查通过：296 张图片均已由 ImageFolder 实际解码。')

## 6. 设置训练参数

默认是 v4 正式训练 1000 步，并使用全新的 Google Drive 输出目录，不会恢复旧版 v3 或 `sd15_light_effect_lora_full` 的检查点。第一次只想确认流程能否跑通，可以把 `TRAINING_MODE` 改为 `"smoke"`，它只训练 50 步，但其结果不能作为最终模型。

训练保持完整宽幅构图，不再把约 3.3:1 的光效图随机裁成 512×512 方图。默认训练尺寸为 768×232（宽高均可被 8 整除），像素总量低于 512×512，适合 Colab T4，同时能让 LoRA 学习完整的左右颜色布局。

In [ ]:
BASE_MODEL = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
BASE_MODEL_REVISION = '451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
TRAINING_RUN = 'v4'
TRAINING_MODE = 'full'  # 可选：'full' 或 'smoke'
assert TRAINING_MODE in {'full', 'smoke'}
MAX_TRAIN_STEPS = 1000 if TRAINING_MODE == 'full' else 50
TRAIN_WIDTH = 768
TRAIN_HEIGHT = 232
assert TRAIN_WIDTH % 8 == 0 and TRAIN_HEIGHT % 8 == 0
OUTPUT_NAME = f'sd15_light_effect_lora_{TRAINING_RUN}' if TRAINING_MODE == 'full' else f'sd15_light_effect_lora_{TRAINING_RUN}_smoke'
OUTPUT_DIR = PROJECT_DIR / OUTPUT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'基础模型：{BASE_MODEL}')
print(f'训练版本：{TRAINING_RUN}')
print(f'训练模式：{TRAINING_MODE}')
print(f'训练步数：{MAX_TRAIN_STEPS}')
print(f'宽幅训练尺寸：{TRAIN_WIDTH} × {TRAIN_HEIGHT}')
print(f'云端输出：{OUTPUT_DIR}')

## 7. 开始训练

训练会每 250 步保存一次检查点。程序只会在新的 v4 输出目录中寻找检查点，不会恢复旧版 v3 或 full 目录。若 v4 训练中断，重新连接后从头运行前面的格子，本格会自动从 v4 最新检查点继续。不要关闭网页；可以偶尔查看日志。首次运行还会下载约数 GB 的基础模型。

In [ ]:
from collections import deque
import os
import subprocess

OFFICIAL_TRAIN_SCRIPT = Path('/content/hf_diffusers_repo/examples/text_to_image/train_text_to_image_lora.py')
TRAIN_SCRIPT = Path('/content/train_text_to_image_lora_panorama.py')
assert OFFICIAL_TRAIN_SCRIPT.is_file(), '官方训练脚本不存在，请重新运行第 4 步。'

# 固定版本脚本默认 Resize 后 RandomCrop 成方图。只替换图像 transform，
# 保留官方 v0.39.0 的优化器、检查点和 LoRA 保存逻辑。
script_source = OFFICIAL_TRAIN_SCRIPT.read_text(encoding='utf-8')
script_lines = script_source.splitlines()
resize_matches = [index for index, line in enumerate(script_lines) if 'transforms.Resize(args.resolution, interpolation=interpolation)' in line]
crop_matches = [index for index, line in enumerate(script_lines) if 'transforms.CenterCrop(args.resolution)' in line and 'transforms.RandomCrop(args.resolution)' in line]
assert len(resize_matches) == 1 and len(crop_matches) == 1, '官方脚本 transform 结构发生变化，请检查固定版本。'
resize_index, crop_index = resize_matches[0], crop_matches[0]
resize_indent = script_lines[resize_index][:-len(script_lines[resize_index].lstrip())]
crop_indent = script_lines[crop_index][:-len(script_lines[crop_index].lstrip())]
script_lines[resize_index] = f'{resize_indent}transforms.Resize(({TRAIN_HEIGHT}, {TRAIN_WIDTH}), interpolation=interpolation),'
script_lines[crop_index] = f'{crop_indent}transforms.Lambda(lambda image: image),'
TRAIN_SCRIPT.write_text('\n'.join(script_lines) + '\n', encoding='utf-8')
print(f'已生成宽幅训练脚本：{TRAIN_SCRIPT}')
cmd = [
    'accelerate', 'launch',
    '--num_processes=1', '--num_machines=1', '--dynamo_backend=no',
    '--mixed_precision=fp16', str(TRAIN_SCRIPT),
    '--mixed_precision', 'fp16',
    '--pretrained_model_name_or_path', BASE_MODEL,
    '--revision', BASE_MODEL_REVISION,
    '--train_data_dir', str(DATA_ROOT / 'train'),
    '--cache_dir', str(WORK_DIR / 'training_cache_v4'),
    '--image_column', 'image',
    '--caption_column', 'text',
    '--resolution', '512',  # 解析器兼容参数；实际 transform 使用 TRAIN_WIDTH/HEIGHT
    '--train_batch_size', '1',
    '--gradient_accumulation_steps', '4',
    '--gradient_checkpointing',
    '--learning_rate', '1e-4',
    '--lr_scheduler', 'constant',
    '--lr_warmup_steps', '0',
    '--max_train_steps', str(MAX_TRAIN_STEPS),
    '--checkpointing_steps', '250',
    '--checkpoints_total_limit', '3',
    '--dataloader_num_workers', '2',
    '--rank', '8',
    '--seed', '20260719',
    '--report_to', 'tensorboard',
    '--output_dir', str(OUTPUT_DIR),
]

existing_checkpoints = list(OUTPUT_DIR.glob('checkpoint-*'))
if existing_checkpoints:
    cmd.extend(['--resume_from_checkpoint', 'latest'])
    print('检测到已有检查点，将自动继续训练。')

env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('即将执行：')
print(' '.join(cmd))
recent_output = deque(maxlen=80)
process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    recent_output.append(line)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'训练脚本退出代码：{return_code}\n\n最后的底层日志：\n' + ''.join(recent_output)
    )
print('训练脚本正常完成。')

## 8. 确认并整理最终权重

训练成功后，官方脚本会输出 `pytorch_lora_weights.safetensors`。下面再复制一份并改成项目里更容易理解的名字 `light_effect_lora.safetensors`。

In [ ]:
import json
import shutil

OFFICIAL_WEIGHT = OUTPUT_DIR / 'pytorch_lora_weights.safetensors'
FINAL_WEIGHT = OUTPUT_DIR / 'light_effect_lora.safetensors'
assert OFFICIAL_WEIGHT.exists(), '没有找到权重文件，请回看训练格是否报错。'
shutil.copy2(OFFICIAL_WEIGHT, FINAL_WEIGHT)

training_config = {
    'base_model': BASE_MODEL,
    'base_model_revision': BASE_MODEL_REVISION,
    'training_run': TRAINING_RUN,
    'training_mode': TRAINING_MODE,
    'dataset_zip_sha256': EXPECTED_ZIP_SHA256,
    'max_train_steps': MAX_TRAIN_STEPS,
    'train_width': TRAIN_WIDTH,
    'train_height': TRAIN_HEIGHT,
    'training_aspect_ratio': TRAIN_WIDTH / TRAIN_HEIGHT,
    'train_count': 254,
    'validation_count': 42,
    'split_strategy': 'source-stratified, recipe/scene-group isolated',
    'lora_rank': 8,
    'learning_rate': 1e-4,
    'seed': 20260719,
}
with (OUTPUT_DIR / 'training_config.json').open('w', encoding='utf-8') as file:
    json.dump(training_config, file, ensure_ascii=False, indent=2)

size_mb = FINAL_WEIGHT.stat().st_size / 1024**2
print(f'最终权重：{FINAL_WEIGHT}')
print(f'文件大小：{size_mb:.1f} MB')

## 9. 生成三张验证图

这一步会加载基础模型和刚训练的 LoRA，输出 1024×320 的宽幅光效图，并保存到 Google Drive。

In [ ]:
import gc
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

gc.collect()
torch.cuda.empty_cache()
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16,
    use_safetensors=True,
    safety_checker=None,
    requires_safety_checker=False,
).to('cuda')
pipe.load_lora_weights(str(OUTPUT_DIR), weight_name='pytorch_lora_weights.safetensors')
pipe.enable_attention_slicing()

validation_prompts = [
    'Warm pale yellow to soft orange gradient with a gentle glow, creating a cozy and inviting atmosphere, smooth luminous light texture, no objects',
    'Soft gradient lighting, warm sunset hues transitioning from amber to soft magenta, cozy and relaxing atmosphere, cinematic lighting, smooth luminous light texture, no objects',
    'Calm pale blue and peach sunrise gradient, soft diffused clouds, bright airy atmosphere, smooth luminous light texture, no objects',
]
negative_prompt = 'text, logo, watermark, people, furniture, lamp, dark green, black background, hard edges, noise'
VALIDATION_DIR = OUTPUT_DIR / 'validation_images'
VALIDATION_DIR.mkdir(exist_ok=True)
images = []
for index, prompt in enumerate(validation_prompts, start=1):
    generator = torch.Generator(device='cuda').manual_seed(20260719 + index)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=1024,
        height=320,
        num_inference_steps=30,
        guidance_scale=7.0,
        generator=generator,
    ).images[0]
    output_path = VALIDATION_DIR / f'validation_{index:02d}.png'
    image.save(output_path)
    images.append(image)

fig, axes = plt.subplots(3, 1, figsize=(16, 8))
for axis, image, prompt in zip(axes, images, validation_prompts):
    axis.imshow(image)
    axis.set_title(prompt, fontsize=9)
    axis.axis('off')
plt.tight_layout()
plt.show()
print(f'验证图已保存到：{VALIDATION_DIR}')

## 10. 下载权重到电脑（可选）

权重已经在 Google Drive 中，因此这一步不是必须的。浏览器可能会询问是否允许下载。

In [ ]:
from google.colab import files
files.download(str(FINAL_WEIGHT))

## 完成标准

你应当在 Google Drive 的 `lighting_lora/sd15_light_effect_lora_v4/` 中看到：

- `light_effect_lora.safetensors`：模块三要加载的最终 LoRA 权重
- `pytorch_lora_weights.safetensors`：相同权重的官方默认文件名
- `training_config.json`：本次训练参数
- `validation_images/`：三张测试图
- `checkpoint-*`：中途检查点，可用于断点续训